In [1]:
!pip install -r requirements.txt

Processing /wheels/flash_attn-2.6.3-cp310-cp310-linux_x86_64.whl (from -r requirements.txt (line 39))
flash_attn is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 3.1 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 3.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 4.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 4.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 4.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 7.1 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 6.0 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: tokenizers91m╸━━━━━━━━━━━━━━━━━━━ 20/39 [markdown-it-py]lt]
    Found existing installation: tokenizers 0.21.4━━━━━━━━━━━━ 20/39 [markdown-it-py]
  

In [2]:
!pip install langchain-huggingface

In [1]:
import os
import dotenv
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain.agents import create_agent
from langchain.tools import tool
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate, SystemMessagePromptTemplate
from peft import PeftModel
import torch
from langchain.tools import tool

/usr/local/lib/python3.10/dist-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
model_name = 'mistralai/Mistral-7B-Instruct-v0.3'
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

# гарантируем eos_token_id
if tokenizer.eos_token_id is None and tokenizer.eos_token is not None:
    tokenizer.eos_token_id = tokenizer.convert_tokens_to_ids(tokenizer.eos_token)

foundation_model = AutoModelForCausalLM.from_pretrained(model_name,
                                                        device_map={"": 0},
                                                        torch_dtype=torch.float16,
                                                        attn_implementation="flash_attention_2")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [3]:
lora_model = PeftModel.from_pretrained(foundation_model, './peft_lab_outputs/lora_adapter_3')
lora_model.eval()

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32768, 4096)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralFlashAttention2(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): 

In [4]:
merged_model = lora_model.merge_and_unload()

In [5]:
@tool
def get_name():
    '''
    Get my name
    '''
    return "Ivan"

In [6]:
text_generation_pipeline = pipeline(
    'text-generation',
    model=merged_model,#foundation_model,#merged_model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.5,
    pad_token_id=tokenizer.eos_token_id
)

In [7]:
llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

In [8]:
chat_model = ChatHuggingFace(llm=llm)

In [9]:
chat_model_with_tools = chat_model.bind_tools([get_name])

In [10]:
tools_desc = """
You have access to a tool:

Tool name: get_name
Description: Get my name
Arguments: none

When you need the user's name, respond ONLY with a JSON object:
{"tool_call": {"name": "get_name", "arguments": {}}}

Otherwise respond ONLY with:
{"final": "..."}
"""

messages = [
    {"role": "system", "content": tools_desc},
    {"role": "user", "content": "What is my name?"}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
out = text_generation_pipeline(prompt)[0]["generated_text"]
print(out[len(prompt):])

 To find out your name using this instruction and its associated tools in Python, we can create an if-else statement. If there are no arguments provided when calling `get_user()`, it means that I am being asked for information about myself (my own identity). In such cases, since our goal here is not just getting any random string but specifically obtaining one person’s unique identifier - their username or handle on Discord/Twitch etc., let us use 'username'. We will then return {'final': self._bot.command('help')['info']['author']}. This command retrieves help documentation from discordpy library which includes author details like bot creator names & handles among other things; however, as per instructions given above only specific parts of these docstrings should be extracted i.e.: [0] refers to first element within square brackets while [] denotes indexing without specifying position inside list(string) so ['info'][1][2]. The resulting output would look something like this:
{'final'

In [11]:
prompt = ChatPromptTemplate.from_messages(
        [
            SystemMessagePromptTemplate.from_template("""
You are helpful assistant
            
            You have access to a tool:

Tool name: get_name
Description: Get my name
Arguments: none

When you need the user's name, respond ONLY with a JSON object:
{{"tool_call": {{"name": "get_name", "arguments": {{}}}}}}

Otherwise respond ONLY with:
{{"final": "..."}}

If you are using tools, answer only with json in format upper.
            """),
            HumanMessagePromptTemplate.from_template("""
Answer the following questions as best you can.
Question: {input}
""")
        ]
    )

In [12]:
agent = create_agent(model = chat_model_with_tools, tools=[get_name])

In [13]:
messages = prompt.invoke({'input': 'What is my name?'}).to_messages()

In [14]:
res = agent.invoke({'messages': messages})

In [15]:
res

{'messages': [SystemMessage(content='\nYou are helpful assistant\n            \n            You have access to a tool:\n\nTool name: get_name\nDescription: Get my name\nArguments: none\n\nWhen you need the user\'s name, respond ONLY with a JSON object:\n{"tool_call": {"name": "get_name", "arguments": {}}}\n\nOtherwise respond ONLY with:\n{"final": "..."}\n\nIf you are using tools, answer only with json in format upper.\n            ', additional_kwargs={}, response_metadata={}, id='7ebc48f3-9d39-49a4-8ba4-3964cd13b3df'),
  HumanMessage(content='\nAnswer the following questions as best you can.\nQuestion: What is my name?\n', additional_kwargs={}, response_metadata={}, id='9cde2be7-28b1-46f1-86ae-cab9e0efe141'),
  AIMessage(content='<s>[INST] \nYou are helpful assistant\n            \n            You have access to a tool:\n\nTool name: get_name\nDescription: Get my name\nArguments: none\n\nWhen you need the user\'s name, respond ONLY with a JSON object:\n{"tool_call": {"name": "get_nam